# 1. Propósito y límites

Este notebook analiza exhaustivamente la corrección completa del primer pivote con el AST no conmutativo de la paquetería. Conserva el orden, no mueve $R_1$ y distingue igualdad exacta, sustitución formal y equivalencia módulo compactos. No demuestra compactitud, pertenencia a una álgebra, admisibilidad Mellin ni Fredholmness.

In [1]:
from IPython.display import Markdown, Math, display
import symbolic_operator_calculus as soc

correction = soc.build_complete_correction_derivation()
trace = soc.build_complete_correction_expansion_trace()
assert trace.counts == (1, 4, 4, 16)
display(Markdown("**Frontera lógica cargada:** B1 es módulo compactos y B2/B3 son exactas."))

**Frontera lógica cargada:** B1 es módulo compactos y B2/B3 son exactas.

## 2. Estado de Fase N

La Fase N obtuvo la identidad formal exacta del pivote y después introdujo dos modelos Wiener--Hopf mediante reglas suministradas módulo compactos. Esa última relación permanece `UNCERTIFIED`.

In [2]:
phase_n = correction.phase_n_trace
assert correction.pivot_relation.left is soc.N2_first
assert correction.rules[0].relation_kind is soc.DerivationRelationKind.MOD_COMPACT_EQUIVALENCE
assert correction.rules[0].certification_status is soc.RuleCertificationStatus.UNCERTIFIED
display(Math(r"N_{2}^{(1)} \simeq N_2+\mathcal C_2"))

<IPython.core.display.Math object>

## 3. Definición de $\mathcal C_2$

La definición estructural siguiente reutiliza exactamente la corrección producida en Fase N; no crea un segundo AST.

In [3]:
original_latex = soc.render_original_complete_correction_latex(correction)
assert correction.correction_definition.right == soc.complete_correction_expression()
display(Math(original_latex))

<IPython.core.display.Math object>

## 4. Normalización exacta de $Z_2$ y $Z_1$

Se aplican únicamente $\widetilde V_{\alpha_k}=Z_kU_k$, $G_k=Z_k\widehat G_k$ y la invertibilidad de $Z_k$. No se conmuta con ningún factor posterior.

In [4]:
rendered_steps = {
    step.key: step
    for step in soc.render_complete_correction_derivation_latex(correction).steps
}
for key in ("B2_left_normalization", "B3_right_normalization"):
    display(Math(rendered_steps[key].latex))
assert correction.left_normalization_rule.certification_status is soc.RuleCertificationStatus.CERTIFIED_EXACT
assert correction.right_normalization_rule.certification_status is soc.RuleCertificationStatus.CERTIFIED_EXACT

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 5. Forma factorizada normalizada

La igualdad en esta celda es exacta dentro del modelo estructural de $\mathcal C_2$; la conexión global con $N_2^{(1)}$ sigue siendo módulo compactos.

In [5]:
c0_latex = soc.render_complete_correction_expansion_latex(trace, "C0")
assert trace.counts[0] == 1
display(Math(c0_latex))

<IPython.core.display.Math object>

## 6. Expansión de diferencias

Solo se distribuyen $U_2-\widehat G_2$ y $U_1-\widehat G_1$.

In [6]:
c1_latex = soc.render_complete_correction_expansion_latex(trace, "C1")
assert len(trace.c1.terms) == 4
assert tuple(term.coefficient for term in trace.c1.terms) == (1, -1, -1, 1)
display(Math(c1_latex))

<IPython.core.display.Math object>

## 7. Expansión de auxiliares

Solo se sustituyen $T_{1,-}=U_1^{-1}P^++P^-$ y $T_{2,-}=U_2^{-1}P^++P^-$. Las diferencias permanecen agrupadas.

In [7]:
c2_latex = soc.render_complete_correction_expansion_latex(trace, "C2")
assert len(trace.c2) == 4
assert all(item.coefficient == 1 for item in trace.c2)
display(Math(c2_latex))

<IPython.core.display.Math object>

## 8. Expansión completa de 16 términos

Se expanden las dos diferencias y los dos auxiliares, sin distribuir internamente los bloques Wiener--Hopf ni $R_1$.

In [8]:
c3_latex = soc.render_complete_correction_expansion_latex(trace, "C3")
expected_ids = tuple(f"C2-T{index:02d}" for index in range(1, 17))
assert tuple(record.identifier for record in trace.terms) == expected_ids
assert tuple(record.coefficient for record in trace.terms) == (1, -1, -1, 1) * 4
assert all(record.term.product.factors.count(soc.R1) == 1 for record in trace.terms)
assert len({record.term.product for record in trace.terms}) == 16
display(Math(c3_latex))

<IPython.core.display.Math object>

## 9. Catálogo de términos

La firma conserva factor concreto, clase declarada, rama, polaridad, relación y posición. Las clases son metadatos, no pruebas analíticas.

In [9]:
catalog_markdown = soc.render_correction_term_catalog_markdown(trace)
assert catalog_markdown.count("C2-T") == 16
display(Markdown(catalog_markdown))

| ID | Signo | Factores ordenados | Firma | Motifs | Interfaces abiertas |
|---|---:|---|---|---|---|
| C2-T01 | + | \(U_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,U_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}\) | 1:U2[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:U1_inverse[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 4:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] → 5:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 6:U1[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 7:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 8:U2_inverse[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 9:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED) | dilation_x_wiener_hopf: U2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wminus_21→U1_inverse (SOURCE_VERIFIED_RULE_MISSING), cauchy_x_r1: P_plus→R1 (SOURCE_UNVERIFIED), r1_x_dilation: R1→U1 (SOURCE_UNVERIFIED), dilation_x_wiener_hopf: U1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wplus_12→U2_inverse (SOURCE_VERIFIED_RULE_MISSING) |
| C2-T02 | - | \(U_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}\) | 1:U2[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:U1_inverse[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 4:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] → 5:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 6:Ghat1[NormalizedCoefficient;branch=1;polarity=None;EXACT_EQUALITY] → 7:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 8:U2_inverse[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 9:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED) | dilation_x_wiener_hopf: U2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wminus_21→U1_inverse (SOURCE_VERIFIED_RULE_MISSING), cauchy_x_r1: P_plus→R1 (SOURCE_UNVERIFIED), r1_x_multiplication: R1→Ghat1 (SOURCE_UNVERIFIED), multiplication_x_wiener_hopf: Ghat1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wplus_12→U2_inverse (SOURCE_VERIFIED_RULE_MISSING) |
| C2-T03 | - | \(\widehat G_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,U_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}\) | 1:Ghat2[NormalizedCoefficient;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:U1_inverse[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 4:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] → 5:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 6:U1[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 7:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 8:U2_inverse[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 9:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED) | multiplication_x_wiener_hopf: Ghat2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wminus_21→U1_inverse (SOURCE_VERIFIED_RULE_MISSING), cauchy_x_r1: P_plus→R1 (SOURCE_UNVERIFIED), r1_x_dilation: R1→U1 (SOURCE_UNVERIFIED), dilation_x_wiener_hopf: U1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wplus_12→U2_inverse (SOURCE_VERIFIED_RULE_MISSING) |
| C2-T04 | + | \(\widehat G_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}\) | 1:Ghat2[NormalizedCoefficient;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:U1_inverse[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 4:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] → 5:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 6:Ghat1[NormalizedCoefficient;branch=1;polarity=None;EXACT_EQUALITY] → 7:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 8:U2_inverse[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 9:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED) | multiplication_x_wiener_hopf: Ghat2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wminus_21→U1_inverse (SOURCE_VERIFIED_RULE_MISSING), cauchy_x_r1: P_plus→R1 (SOURCE_UNVERIFIED), r1_x_multiplication: R1→Ghat1 (SOURCE_UNVERIFIED), multiplication_x_wiener_hopf: Ghat1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wplus_12→U2_inverse (SOURCE_VERIFIED_RULE_MISSING) |
| C2-T05 | + | \(U_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,U_{1}\,W^+_{1,2}\,P^{-}\) | 1:U2[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:U1_inverse[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 4:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] → 5:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 6:U1[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 7:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 8:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED) | dilation_x_wiener_hopf: U2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wminus_21→U1_inverse (SOURCE_VERIFIED_RULE_MISSING), cauchy_x_r1: P_plus→R1 (SOURCE_UNVERIFIED), r1_x_dilation: R1→U1 (SOURCE_UNVERIFIED), dilation_x_wiener_hopf: U1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wplus_12→P_minus (NO_RULE_FOUND) |
| C2-T06 | - | \(U_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,P^{-}\) | 1:U2[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:U1_inverse[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 4:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] → 5:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 6:Ghat1[NormalizedCoefficient;branch=1;polarity=None;EXACT_EQUALITY] → 7:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 8:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED) | dilation_x_wiener_hopf: U2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wminus_21→U1_inverse (SOURCE_VERIFIED_RULE_MISSING), cauchy_x_r1: P_plus→R1 (SOURCE_UNVERIFIED), r1_x_multiplication: R1→Ghat1 (SOURCE_UNVERIFIED), multiplication_x_wiener_hopf: Ghat1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wplus_12→P_minus (NO_RULE_FOUND) |
| C2-T07 | - | \(\widehat G_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,U_{1}\,W^+_{1,2}\,P^{-}\) | 1:Ghat2[NormalizedCoefficient;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:U1_inverse[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 4:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] → 5:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 6:U1[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 7:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 8:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED) | multiplication_x_wiener_hopf: Ghat2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wminus_21→U1_inverse (SOURCE_VERIFIED_RULE_MISSING), cauchy_x_r1: P_plus→R1 (SOURCE_UNVERIFIED), r1_x_dilation: R1→U1 (SOURCE_UNVERIFIED), dilation_x_wiener_hopf: U1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wplus_12→P_minus (NO_RULE_FOUND) |
| C2-T08 | + | \(\widehat G_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,P^{-}\) | 1:Ghat2[NormalizedCoefficient;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:U1_inverse[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 4:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] → 5:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 6:Ghat1[NormalizedCoefficient;branch=1;polarity=None;EXACT_EQUALITY] → 7:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 8:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED) | multiplication_x_wiener_hopf: Ghat2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wminus_21→U1_inverse (SOURCE_VERIFIED_RULE_MISSING), cauchy_x_r1: P_plus→R1 (SOURCE_UNVERIFIED), r1_x_multiplication: R1→Ghat1 (SOURCE_UNVERIFIED), multiplication_x_wiener_hopf: Ghat1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wplus_12→P_minus (NO_RULE_FOUND) |
| C2-T09 | + | \(U_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,U_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}\) | 1:U2[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] → 4:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 5:U1[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 6:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 7:U2_inverse[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 8:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED) | dilation_x_wiener_hopf: U2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wminus_21→P_minus (NO_RULE_FOUND), cauchy_x_r1: P_minus→R1 (SOURCE_UNVERIFIED), r1_x_dilation: R1→U1 (SOURCE_UNVERIFIED), dilation_x_wiener_hopf: U1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wplus_12→U2_inverse (SOURCE_VERIFIED_RULE_MISSING) |
| C2-T10 | - | \(U_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}\) | 1:U2[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] → 4:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 5:Ghat1[NormalizedCoefficient;branch=1;polarity=None;EXACT_EQUALITY] → 6:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 7:U2_inverse[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 8:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED) | dilation_x_wiener_hopf: U2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wminus_21→P_minus (NO_RULE_FOUND), cauchy_x_r1: P_minus→R1 (SOURCE_UNVERIFIED), r1_x_multiplication: R1→Ghat1 (SOURCE_UNVERIFIED), multiplication_x_wiener_hopf: Ghat1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wplus_12→U2_inverse (SOURCE_VERIFIED_RULE_MISSING) |
| C2-T11 | - | \(\widehat G_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,U_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}\) | 1:Ghat2[NormalizedCoefficient;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] → 4:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 5:U1[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 6:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 7:U2_inverse[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 8:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED) | multiplication_x_wiener_hopf: Ghat2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wminus_21→P_minus (NO_RULE_FOUND), cauchy_x_r1: P_minus→R1 (SOURCE_UNVERIFIED), r1_x_dilation: R1→U1 (SOURCE_UNVERIFIED), dilation_x_wiener_hopf: U1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wplus_12→U2_inverse (SOURCE_VERIFIED_RULE_MISSING) |
| C2-T12 | + | \(\widehat G_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}\) | 1:Ghat2[NormalizedCoefficient;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] → 4:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 5:Ghat1[NormalizedCoefficient;branch=1;polarity=None;EXACT_EQUALITY] → 6:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 7:U2_inverse[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 8:P_plus[CauchyFactorPlus;branch=None;polarity=+;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED), mixed_cauchy_dilation (EXACT_RECOGNIZED) | multiplication_x_wiener_hopf: Ghat2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wminus_21→P_minus (NO_RULE_FOUND), cauchy_x_r1: P_minus→R1 (SOURCE_UNVERIFIED), r1_x_multiplication: R1→Ghat1 (SOURCE_UNVERIFIED), multiplication_x_wiener_hopf: Ghat1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_dilation: Wplus_12→U2_inverse (SOURCE_VERIFIED_RULE_MISSING) |
| C2-T13 | + | \(U_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,U_{1}\,W^+_{1,2}\,P^{-}\) | 1:U2[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] → 4:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 5:U1[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 6:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 7:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED) | dilation_x_wiener_hopf: U2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wminus_21→P_minus (NO_RULE_FOUND), cauchy_x_r1: P_minus→R1 (SOURCE_UNVERIFIED), r1_x_dilation: R1→U1 (SOURCE_UNVERIFIED), dilation_x_wiener_hopf: U1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wplus_12→P_minus (NO_RULE_FOUND) |
| C2-T14 | - | \(U_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,P^{-}\) | 1:U2[DilationOperator;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] → 4:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 5:Ghat1[NormalizedCoefficient;branch=1;polarity=None;EXACT_EQUALITY] → 6:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 7:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED) | dilation_x_wiener_hopf: U2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wminus_21→P_minus (NO_RULE_FOUND), cauchy_x_r1: P_minus→R1 (SOURCE_UNVERIFIED), r1_x_multiplication: R1→Ghat1 (SOURCE_UNVERIFIED), multiplication_x_wiener_hopf: Ghat1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wplus_12→P_minus (NO_RULE_FOUND) |
| C2-T15 | - | \(\widehat G_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,U_{1}\,W^+_{1,2}\,P^{-}\) | 1:Ghat2[NormalizedCoefficient;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] → 4:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 5:U1[DilationOperator;branch=1;polarity=None;EXACT_EQUALITY] → 6:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 7:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED) | multiplication_x_wiener_hopf: Ghat2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wminus_21→P_minus (NO_RULE_FOUND), cauchy_x_r1: P_minus→R1 (SOURCE_UNVERIFIED), r1_x_dilation: R1→U1 (SOURCE_UNVERIFIED), dilation_x_wiener_hopf: U1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wplus_12→P_minus (NO_RULE_FOUND) |
| C2-T16 | + | \(\widehat G_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,P^{-}\) | 1:Ghat2[NormalizedCoefficient;branch=2;polarity=None;EXACT_EQUALITY] → 2:Wminus_21[LocalizedWienerHopfMinus;branch=2;polarity=-;MOD_COMPACT_EQUIVALENCE] → 3:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] → 4:R1[MellinPDORegularizer;branch=1;polarity=None;FORMAL_SUBSTITUTION] → 5:Ghat1[NormalizedCoefficient;branch=1;polarity=None;EXACT_EQUALITY] → 6:Wplus_12[LocalizedWienerHopfPlus;branch=1;polarity=+;MOD_COMPACT_EQUIVALENCE] → 7:P_minus[CauchyFactorMinus;branch=None;polarity=-;EXACT_EQUALITY] | normalized_left_shift (SUPPLIED_MOD_COMPACT), left_auxiliary_piece (EXACT_RECOGNIZED), mellin_regularizer_core (UNRESOLVED), normalized_right_shift (SUPPLIED_MOD_COMPACT), right_auxiliary_piece (EXACT_RECOGNIZED) | multiplication_x_wiener_hopf: Ghat2→Wminus_21 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wminus_21→P_minus (NO_RULE_FOUND), cauchy_x_r1: P_minus→R1 (SOURCE_UNVERIFIED), r1_x_multiplication: R1→Ghat1 (SOURCE_UNVERIFIED), multiplication_x_wiener_hopf: Ghat1→Wplus_12 (SOURCE_VERIFIED_RULE_MISSING), wiener_hopf_x_cauchy: Wplus_12→P_minus (NO_RULE_FOUND) |

## 10. Motifs reconocidos

Solo se detectan subproductos contiguos. Ningún motif exige conmutar a través de un Wiener--Hopf o de $R_1$.

In [10]:
motifs_markdown = soc.render_correction_motif_frequency_markdown(trace)
assert all(
    motif.status is not soc.MotifStatus.FORBIDDEN_WITHOUT_COMMUTATION
    for record in trace.terms
    for motif in record.motifs
)
display(Markdown(motifs_markdown))

| Motif | Estado | Frecuencia |
|---|---|---:|
| `left_auxiliary_piece` | EXACT_RECOGNIZED | 16 |
| `mellin_regularizer_core` | UNRESOLVED | 16 |
| `mixed_cauchy_dilation` | EXACT_RECOGNIZED | 16 |
| `normalized_left_shift` | SUPPLIED_MOD_COMPACT | 16 |
| `normalized_right_shift` | SUPPLIED_MOD_COMPACT | 16 |
| `right_auxiliary_piece` | EXACT_RECOGNIZED | 16 |

## 11. Matriz de interfaces

Los estados conservadores registran ausencia de regla o evidencia; no rellenan huecos con inferencias de clase.

In [11]:
interfaces_markdown = soc.render_correction_interface_matrix_markdown(trace)
assert all(
    row.status not in (soc.InterfaceStatus.CERTIFIED_EXACT, soc.InterfaceStatus.CERTIFIED_MOD_COMPACT)
    for row in trace.interface_matrix
    if row.term_count
)
display(Markdown(interfaces_markdown))

| Interfaz | Nº de términos | Regla existente | Fuente | Estado |
|---|---:|---|---|---|
| multiplicación × Wiener--Hopf | 16 | no implemented semiproduct rule | paper exact normalization; closure rule absent | SOURCE_VERIFIED_RULE_MISSING |
| dilatación × Wiener--Hopf | 16 | no implemented closure rule | paper exact normalization; closure rule absent | SOURCE_VERIFIED_RULE_MISSING |
| Wiener--Hopf × dilatación | 16 | exact auxiliary expansion only | T_{k,-} definition; semiproduct rule absent | SOURCE_VERIFIED_RULE_MISSING |
| Wiener--Hopf × Cauchy factor | 16 | none | no rule found in the implemented API | NO_RULE_FOUND |
| Cauchy factor × R_1 | 16 | none verified | KKL 2014 source verification pending | SOURCE_UNVERIFIED |
| Wiener--Hopf × R_1 | 0 | not contiguous in C3 | structural inspection | NOT_APPLICABLE |
| R_1 × multiplicación | 8 | none verified | Mellin semiproduct source verification pending | SOURCE_UNVERIFIED |
| R_1 × dilatación | 8 | none verified | Mellin-dilation source verification pending | SOURCE_UNVERIFIED |
| R_1 × Wiener--Hopf | 0 | not contiguous in C3 | structural inspection | NOT_APPLICABLE |
| Wiener--Hopf × auxiliar | 8 | T_{k,-}=U_k^{-1}P^+ + P^- | exact supplied auxiliary identity; interface rule absent | SOURCE_VERIFIED_RULE_MISSING |
| productos con cutoffs/localización explícitos | 0 | no explicit cutoff atom in C3 | localized status is metadata on W factors | NOT_APPLICABLE |

## 12. Fuentes externas consultadas

La consulta fue de solo lectura y respetó el contrato del índice de literatura. Los resultados verificados se aplican aquí únicamente como adaptaciones aún no demostradas.

In [12]:
reference_summary = """| Fuente | Evidencia verificada | Aplicación actual |
|---|---|---|
| KKL2014Regularization, Thm. 5.8 | impresa 206, PDF 18; símbolo en E-tilde y no anulación requeridos | unproved_adaptation |
| Karlovich2025Cusps, Lemmas 3.1–3.2 y Cor. 3.3 | PDF 8–13; errores transportados explícitos compactos | identificación de bloques pendiente |
| Karlovich2025Cusps, Thms. 6.1–6.2 | PDF 20–25; símbolos y criterio para la álgebra de la fuente | pertenencia de la corrección pendiente |
| Karlovich2017HasemanSO, Thms. 5.6/6.2/7.1 | fuente ambigua, sin PDF resuelto | source verification pending |
| Karlovich2008Nonlocal y KarlovichMonsivais2024MellinPDO | PDFs resueltos, sin registros de teorema verificados | source verification pending |
| artículo 2014 sobre shifts | Thms. 5.2/1.1 leídos, pero sin clave BibTeX canónica | regla certificada bloqueada |
"""
display(Markdown(reference_summary))

| Fuente | Evidencia verificada | Aplicación actual |
|---|---|---|
| KKL2014Regularization, Thm. 5.8 | impresa 206, PDF 18; símbolo en E-tilde y no anulación requeridos | unproved_adaptation |
| Karlovich2025Cusps, Lemmas 3.1–3.2 y Cor. 3.3 | PDF 8–13; errores transportados explícitos compactos | identificación de bloques pendiente |
| Karlovich2025Cusps, Thms. 6.1–6.2 | PDF 20–25; símbolos y criterio para la álgebra de la fuente | pertenencia de la corrección pendiente |
| Karlovich2017HasemanSO, Thms. 5.6/6.2/7.1 | fuente ambigua, sin PDF resuelto | source verification pending |
| Karlovich2008Nonlocal y KarlovichMonsivais2024MellinPDO | PDFs resueltos, sin registros de teorema verificados | source verification pending |
| artículo 2014 sobre shifts | Thms. 5.2/1.1 leídos, pero sin clave BibTeX canónica | regla certificada bloqueada |


## 13. Obligaciones de prueba

Se agregan las obligaciones generadas por los 16 términos y se eliminan únicamente duplicados textuales, sin rebajar su fuerza lógica.

In [13]:
obligation_texts = []
for record in trace.terms:
    for obligation in record.proof_obligations:
        if obligation.statement not in obligation_texts:
            obligation_texts.append(obligation.statement)
assert obligation_texts
assert all(
    obligation.relation_kind is soc.DerivationRelationKind.ANALYTIC_PROOF_OBLIGATION
    for record in trace.terms
    for obligation in record.proof_obligations
)
display(Markdown("\n".join(f"{index}. {text}" for index, text in enumerate(obligation_texts, 1))))

1. Justify the unresolved dilation_x_wiener_hopf interfaces in C2-T01 without commuting factors.
2. Justify the unresolved wiener_hopf_x_dilation interfaces in C2-T01 without commuting factors.
3. Justify the unresolved cauchy_x_r1 interfaces in C2-T01 without commuting factors.
4. Justify the unresolved r1_x_dilation interfaces in C2-T01 without commuting factors.
5. Justify the unresolved dilation_x_wiener_hopf interfaces in C2-T02 without commuting factors.
6. Justify the unresolved wiener_hopf_x_dilation interfaces in C2-T02 without commuting factors.
7. Justify the unresolved cauchy_x_r1 interfaces in C2-T02 without commuting factors.
8. Justify the unresolved r1_x_multiplication interfaces in C2-T02 without commuting factors.
9. Justify the unresolved multiplication_x_wiener_hopf interfaces in C2-T02 without commuting factors.
10. Justify the unresolved multiplication_x_wiener_hopf interfaces in C2-T03 without commuting factors.
11. Justify the unresolved wiener_hopf_x_dilation interfaces in C2-T03 without commuting factors.
12. Justify the unresolved cauchy_x_r1 interfaces in C2-T03 without commuting factors.
13. Justify the unresolved r1_x_dilation interfaces in C2-T03 without commuting factors.
14. Justify the unresolved dilation_x_wiener_hopf interfaces in C2-T03 without commuting factors.
15. Justify the unresolved multiplication_x_wiener_hopf interfaces in C2-T04 without commuting factors.
16. Justify the unresolved wiener_hopf_x_dilation interfaces in C2-T04 without commuting factors.
17. Justify the unresolved cauchy_x_r1 interfaces in C2-T04 without commuting factors.
18. Justify the unresolved r1_x_multiplication interfaces in C2-T04 without commuting factors.
19. Justify the unresolved dilation_x_wiener_hopf interfaces in C2-T05 without commuting factors.
20. Justify the unresolved wiener_hopf_x_dilation interfaces in C2-T05 without commuting factors.
21. Justify the unresolved cauchy_x_r1 interfaces in C2-T05 without commuting factors.
22. Justify the unresolved r1_x_dilation interfaces in C2-T05 without commuting factors.
23. Justify the unresolved wiener_hopf_x_cauchy interfaces in C2-T05 without commuting factors.
24. Justify the unresolved dilation_x_wiener_hopf interfaces in C2-T06 without commuting factors.
25. Justify the unresolved wiener_hopf_x_dilation interfaces in C2-T06 without commuting factors.
26. Justify the unresolved cauchy_x_r1 interfaces in C2-T06 without commuting factors.
27. Justify the unresolved r1_x_multiplication interfaces in C2-T06 without commuting factors.
28. Justify the unresolved multiplication_x_wiener_hopf interfaces in C2-T06 without commuting factors.
29. Justify the unresolved wiener_hopf_x_cauchy interfaces in C2-T06 without commuting factors.
30. Justify the unresolved multiplication_x_wiener_hopf interfaces in C2-T07 without commuting factors.
31. Justify the unresolved wiener_hopf_x_dilation interfaces in C2-T07 without commuting factors.
32. Justify the unresolved cauchy_x_r1 interfaces in C2-T07 without commuting factors.
33. Justify the unresolved r1_x_dilation interfaces in C2-T07 without commuting factors.
34. Justify the unresolved dilation_x_wiener_hopf interfaces in C2-T07 without commuting factors.
35. Justify the unresolved wiener_hopf_x_cauchy interfaces in C2-T07 without commuting factors.
36. Justify the unresolved multiplication_x_wiener_hopf interfaces in C2-T08 without commuting factors.
37. Justify the unresolved wiener_hopf_x_dilation interfaces in C2-T08 without commuting factors.
38. Justify the unresolved cauchy_x_r1 interfaces in C2-T08 without commuting factors.
39. Justify the unresolved r1_x_multiplication interfaces in C2-T08 without commuting factors.
40. Justify the unresolved wiener_hopf_x_cauchy interfaces in C2-T08 without commuting factors.
41. Justify the unresolved dilation_x_wiener_hopf interfaces in C2-T09 without commuting factors.
42. Justify the unresolved wiener_hopf_x_cauchy interfaces in C2-T09 without commuting factors.
43. Justify the unresolved cauchy_x_r1 interfaces in C2-T09 without commuting factors.
44. Justify the unresolved r1_x_dilation interfaces in C2-T09 without commuting factors.
45. Justify the unresolved wiener_hopf_x_dilation interfaces in C2-T09 without commuting factors.
46. Justify the unresolved dilation_x_wiener_hopf interfaces in C2-T10 without commuting factors.
47. Justify the unresolved wiener_hopf_x_cauchy interfaces in C2-T10 without commuting factors.
48. Justify the unresolved cauchy_x_r1 interfaces in C2-T10 without commuting factors.
49. Justify the unresolved r1_x_multiplication interfaces in C2-T10 without commuting factors.
50. Justify the unresolved multiplication_x_wiener_hopf interfaces in C2-T10 without commuting factors.
51. Justify the unresolved wiener_hopf_x_dilation interfaces in C2-T10 without commuting factors.
52. Justify the unresolved multiplication_x_wiener_hopf interfaces in C2-T11 without commuting factors.
53. Justify the unresolved wiener_hopf_x_cauchy interfaces in C2-T11 without commuting factors.
54. Justify the unresolved cauchy_x_r1 interfaces in C2-T11 without commuting factors.
55. Justify the unresolved r1_x_dilation interfaces in C2-T11 without commuting factors.
56. Justify the unresolved dilation_x_wiener_hopf interfaces in C2-T11 without commuting factors.
57. Justify the unresolved wiener_hopf_x_dilation interfaces in C2-T11 without commuting factors.
58. Justify the unresolved multiplication_x_wiener_hopf interfaces in C2-T12 without commuting factors.
59. Justify the unresolved wiener_hopf_x_cauchy interfaces in C2-T12 without commuting factors.
60. Justify the unresolved cauchy_x_r1 interfaces in C2-T12 without commuting factors.
61. Justify the unresolved r1_x_multiplication interfaces in C2-T12 without commuting factors.
62. Justify the unresolved wiener_hopf_x_dilation interfaces in C2-T12 without commuting factors.
63. Justify the unresolved dilation_x_wiener_hopf interfaces in C2-T13 without commuting factors.
64. Justify the unresolved wiener_hopf_x_cauchy interfaces in C2-T13 without commuting factors.
65. Justify the unresolved cauchy_x_r1 interfaces in C2-T13 without commuting factors.
66. Justify the unresolved r1_x_dilation interfaces in C2-T13 without commuting factors.
67. Justify the unresolved dilation_x_wiener_hopf interfaces in C2-T14 without commuting factors.
68. Justify the unresolved wiener_hopf_x_cauchy interfaces in C2-T14 without commuting factors.
69. Justify the unresolved cauchy_x_r1 interfaces in C2-T14 without commuting factors.
70. Justify the unresolved r1_x_multiplication interfaces in C2-T14 without commuting factors.
71. Justify the unresolved multiplication_x_wiener_hopf interfaces in C2-T14 without commuting factors.
72. Justify the unresolved multiplication_x_wiener_hopf interfaces in C2-T15 without commuting factors.
73. Justify the unresolved wiener_hopf_x_cauchy interfaces in C2-T15 without commuting factors.
74. Justify the unresolved cauchy_x_r1 interfaces in C2-T15 without commuting factors.
75. Justify the unresolved r1_x_dilation interfaces in C2-T15 without commuting factors.
76. Justify the unresolved dilation_x_wiener_hopf interfaces in C2-T15 without commuting factors.
77. Justify the unresolved multiplication_x_wiener_hopf interfaces in C2-T16 without commuting factors.
78. Justify the unresolved wiener_hopf_x_cauchy interfaces in C2-T16 without commuting factors.
79. Justify the unresolved cauchy_x_r1 interfaces in C2-T16 without commuting factors.
80. Justify the unresolved r1_x_multiplication interfaces in C2-T16 without commuting factors.

## 14. Decisión A/B/C

A exige una representación $\operatorname{Op}(c)+K$ y reglas de semiproducto verificadas. B exige pertenencia demostrada a la álgebra cuspídea y compatibilidad de shifts y $R_1$.

In [14]:
route_decision = "C: insufficient evidence"
assert any(row.status is soc.InterfaceStatus.SOURCE_UNVERIFIED for row in trace.interface_matrix)
assert any(row.status is soc.InterfaceStatus.NO_RULE_FOUND for row in trace.interface_matrix)
display(Markdown(f"**{route_decision}** — faltan reglas centrales para ambas rutas."))

**C: insufficient evidence** — faltan reglas centrales para ambas rutas.

## 15. Exportación LaTeX

El fragmento se compone solo con renderizadores públicos de la paquetería y conserva los cuatro niveles C0/C1/C2/C3.

In [15]:
latex_fragment = "\n\n".join([
    soc.render_complete_correction_derivation_latex(correction).as_latex(),
    c0_latex,
    c1_latex,
    c2_latex,
    c3_latex,
])
assert latex_fragment.count("C2-T") == 16
display(Markdown("```latex\n" + latex_fragment + "\n```"))

```latex
N_{2}^{(1)} \simeq N_{2} + \mathcal C_{2}

\mathcal C_{2} = Z_{2}^{-1}\,\left(\widetilde V_{\alpha_2} - G_{2}\right)\,W^-_{2,1}\,T_{1,-}\,R_{1}\,Z_{1}^{-1}\,\left(\widetilde V_{\alpha_1} - G_{1}\right)\,W^+_{1,2}\,T_{2,-}

Z_{2}^{-1}\,\left(\widetilde V_{\alpha_2} - G_{2}\right) = U_{2} - \widehat G_{2}

Z_{1}^{-1}\,\left(\widetilde V_{\alpha_1} - G_{1}\right) = U_{1} - \widehat G_{1}

\mathcal C_{2}^{\mathrm{norm}} = \left(U_{2} - \widehat G_{2}\right)\,W^-_{2,1}\,T_{1,-}\,R_{1}\,\left(U_{1} - \widehat G_{1}\right)\,W^+_{1,2}\,T_{2,-}

\mathcal C_{2}^{\mathrm{norm}} = \left(U_{2} - \widehat G_{2}\right)\,W^-_{2,1}\,T_{1,-}\,R_{1}\,\left(U_{1} - \widehat G_{1}\right)\,W^+_{1,2}\,T_{2,-}

\mathcal C_{2}^{\mathrm{norm}} = U_{2}\,W^-_{2,1}\,T_{1,-}\,R_{1}\,U_{1}\,W^+_{1,2}\,T_{2,-} - U_{2}\,W^-_{2,1}\,T_{1,-}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,T_{2,-} - \widehat G_{2}\,W^-_{2,1}\,T_{1,-}\,R_{1}\,U_{1}\,W^+_{1,2}\,T_{2,-} + \widehat G_{2}\,W^-_{2,1}\,T_{1,-}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,T_{2,-}

\begin{aligned}\mathcal C_{2}^{\mathrm{norm}} &={}& \left(U_{2} - \widehat G_{2}\right)\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,\left(U_{1} - \widehat G_{1}\right)\,W^+_{1,2}\,U_{2}^{-1}\,P^{+} \\ &&{}+ \left(U_{2} - \widehat G_{2}\right)\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,\left(U_{1} - \widehat G_{1}\right)\,W^+_{1,2}\,P^{-} \\ &&{}+ \left(U_{2} - \widehat G_{2}\right)\,W^-_{2,1}\,P^{-}\,R_{1}\,\left(U_{1} - \widehat G_{1}\right)\,W^+_{1,2}\,U_{2}^{-1}\,P^{+} \\ &&{}+ \left(U_{2} - \widehat G_{2}\right)\,W^-_{2,1}\,P^{-}\,R_{1}\,\left(U_{1} - \widehat G_{1}\right)\,W^+_{1,2}\,P^{-}\end{aligned}

\begin{aligned}\mathcal C_{2}^{\mathrm{norm}} &={}& \underbrace{U_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,U_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}}_{\mathrm{C2-T01}} \\ &&- \underbrace{U_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}}_{\mathrm{C2-T02}} \\ &&- \underbrace{\widehat G_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,U_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}}_{\mathrm{C2-T03}} \\ &&+ \underbrace{\widehat G_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}}_{\mathrm{C2-T04}} \\ &&+ \underbrace{U_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,U_{1}\,W^+_{1,2}\,P^{-}}_{\mathrm{C2-T05}} \\ &&- \underbrace{U_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,P^{-}}_{\mathrm{C2-T06}} \\ &&- \underbrace{\widehat G_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,U_{1}\,W^+_{1,2}\,P^{-}}_{\mathrm{C2-T07}} \\ &&+ \underbrace{\widehat G_{2}\,W^-_{2,1}\,U_{1}^{-1}\,P^{+}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,P^{-}}_{\mathrm{C2-T08}} \\ &&+ \underbrace{U_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,U_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}}_{\mathrm{C2-T09}} \\ &&- \underbrace{U_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}}_{\mathrm{C2-T10}} \\ &&- \underbrace{\widehat G_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,U_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}}_{\mathrm{C2-T11}} \\ &&+ \underbrace{\widehat G_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,U_{2}^{-1}\,P^{+}}_{\mathrm{C2-T12}} \\ &&+ \underbrace{U_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,U_{1}\,W^+_{1,2}\,P^{-}}_{\mathrm{C2-T13}} \\ &&- \underbrace{U_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,P^{-}}_{\mathrm{C2-T14}} \\ &&- \underbrace{\widehat G_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,U_{1}\,W^+_{1,2}\,P^{-}}_{\mathrm{C2-T15}} \\ &&+ \underbrace{\widehat G_{2}\,W^-_{2,1}\,P^{-}\,R_{1}\,\widehat G_{1}\,W^+_{1,2}\,P^{-}}_{\mathrm{C2-T16}}\end{aligned}
```

## 16. Frontera de capacidad de la paquetería

La paquetería certifica la estructura no conmutativa, las normalizaciones exactas suministradas, los conteos y los signos de las expansiones, las firmas y la trazabilidad lógica. No certifica las reglas Wiener--Hopf módulo compactos, cierre analítico, clase Mellin, pertenencia a la álgebra cuspídea ni Fredholmness.

In [16]:
capacity = """**Demostrado formalmente:** orden, signos, normalización exacta de Z, expansión 1/4/4/16, atomicidad de R1, catálogo y determinismo.

**Suministrado pero no certificado:** modelos Wiener--Hopf módulo compactos.

**Abierto:** reglas de interfaces, membresía/cierre, símbolo analítico y cualquier criterio de Fredholmness.
"""
display(Markdown(capacity))

**Demostrado formalmente:** orden, signos, normalización exacta de Z, expansión 1/4/4/16, atomicidad de R1, catálogo y determinismo.

**Suministrado pero no certificado:** modelos Wiener--Hopf módulo compactos.

**Abierto:** reglas de interfaces, membresía/cierre, símbolo analítico y cualquier criterio de Fredholmness.
